# Retail Sales Data Pipeline — Azure Data Engineering

End-to-end Medallion Architecture pipeline using Azure Data Factory, ADLS Gen2,
Azure Databricks, PySpark, Delta Lake, Unity Catalog, and Power BI.

**Flow:** Azure SQL / external customer source → ADF → ADLS Bronze → Databricks/PySpark
→ Silver Delta → Gold Delta → Unity Catalog → Power BI


## 1. Configuration

Storage access is handled through a Unity Catalog External Location backed by an
Azure Managed Identity / Access Connector. No storage account keys or DBFS mounts
are used in this notebook.


In [ ]:
base_path = "abfss://retail@retailstorage2111.dfs.core.windows.net"

bronze_path = f"{base_path}/bronze"
silver_path = f"{base_path}/silver"
gold_path = f"{base_path}/gold"

customer_path = (
    f"{bronze_path}/customer/"
    "manish040596/azure-data-engineer---multi-source/refs/heads/main/"
)


## 2. Read Bronze Layer

ADF ingests transaction, product, and store data as Parquet into ADLS Gen2.
The customer dataset is also stored as Parquet under its source-derived folder path.


In [ ]:
df_transactions = spark.read.parquet(f"{bronze_path}/transaction/")
df_products = spark.read.parquet(f"{bronze_path}/product/")
df_stores = spark.read.parquet(f"{bronze_path}/store/")
df_customers = spark.read.parquet(customer_path)


In [ ]:
print("TRANSACTIONS")
df_transactions.printSchema()

print("\nPRODUCTS")
df_products.printSchema()

print("\nSTORES")
df_stores.printSchema()

print("\nCUSTOMERS")
df_customers.printSchema()


## 3. Silver Layer — Cleaning and Standardization

Standardize data types, retain required customer attributes, and deduplicate customer records
before joining the datasets.


In [ ]:
from pyspark.sql.functions import col

df_transactions_clean = df_transactions.select(
    col("transaction_id").cast("int"),
    col("customer_id").cast("int"),
    col("product_id").cast("int"),
    col("store_id").cast("int"),
    col("quantity").cast("int"),
    col("transaction_date").cast("date")
)

df_products_clean = df_products.select(
    col("product_id").cast("int"),
    col("product_name"),
    col("category"),
    col("price").cast("double")
)

df_stores_clean = df_stores.select(
    col("store_id").cast("int"),
    col("store_name"),
    col("location")
)

df_customers_clean = (
    df_customers.select(
        col("customer_id").cast("int"),
        col("first_name"),
        col("last_name"),
        col("email"),
        col("phone"),
        col("city"),
        col("registration_date").cast("date")
    )
    .dropDuplicates(["customer_id"])
)


## 4. Join and Enrich Silver Dataset

Join transactions with customer, product, and store dimensions and calculate transaction revenue.


In [ ]:
df_silver = (
    df_transactions_clean
    .join(df_customers_clean, "customer_id", "inner")
    .join(df_products_clean, "product_id", "inner")
    .join(df_stores_clean, "store_id", "inner")
    .withColumn("total_amount", col("quantity") * col("price"))
)

display(df_silver)


## 5. Data Quality Validation

Validate row preservation, duplicate transaction IDs, and null transaction IDs before persisting Silver.


In [ ]:
print("Transactions before join:", df_transactions_clean.count())
print("Silver rows after join:", df_silver.count())

print(
    "Duplicate transaction IDs:",
    df_silver.groupBy("transaction_id")
             .count()
             .filter(col("count") > 1)
             .count()
)

print(
    "Null transaction IDs:",
    df_silver.filter(col("transaction_id").isNull()).count()
)


## 6. Persist Silver as Delta and Register in Unity Catalog


In [ ]:
df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_path)


In [ ]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS retail_workspace.default.retail_silver_cleaned
USING DELTA
LOCATION '{silver_path}'
''')

spark.sql('''
SELECT COUNT(*) AS silver_count
FROM retail_workspace.default.retail_silver_cleaned
''').show()


## 7. Gold Layer — Business Aggregations

Create analytics-ready sales metrics by date, product, category, and store.


In [ ]:
from pyspark.sql.functions import sum, countDistinct, avg, round

silver_df = spark.table(
    "retail_workspace.default.retail_silver_cleaned"
)

gold_df = (
    silver_df
    .groupBy(
        "transaction_date",
        "product_id",
        "product_name",
        "category",
        "store_id",
        "store_name",
        "location"
    )
    .agg(
        sum("quantity").alias("total_quantity_sold"),
        round(sum("total_amount"), 2).alias("total_sales_amount"),
        countDistinct("transaction_id").alias("number_of_transactions"),
        round(avg("total_amount"), 2).alias("average_transaction_value")
    )
)

display(gold_df)


In [ ]:
print("Gold rows:", gold_df.count())
gold_df.printSchema()

gold_df.select(
    "total_quantity_sold",
    "total_sales_amount",
    "number_of_transactions",
    "average_transaction_value"
).summary().show()


## 8. Persist Gold as Delta and Register in Unity Catalog


In [ ]:
gold_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(gold_path)


In [ ]:
spark.sql(f'''
CREATE TABLE IF NOT EXISTS retail_workspace.default.retail_gold_sales_summary
USING DELTA
LOCATION '{gold_path}'
''')

spark.sql('''
SELECT COUNT(*) AS gold_count
FROM retail_workspace.default.retail_gold_sales_summary
''').show()


## 9. Business Validation Queries

These queries validate that the Gold layer supports common reporting requirements
before connecting it to Power BI.


In [ ]:
# Top products by revenue
display(
    spark.sql('''
        SELECT
            product_name,
            category,
            ROUND(SUM(total_sales_amount), 2) AS revenue
        FROM retail_workspace.default.retail_gold_sales_summary
        GROUP BY product_name, category
        ORDER BY revenue DESC
    ''')
)


In [ ]:
# Store performance
display(
    spark.sql('''
        SELECT
            store_name,
            location,
            ROUND(SUM(total_sales_amount), 2) AS revenue,
            SUM(total_quantity_sold) AS units_sold
        FROM retail_workspace.default.retail_gold_sales_summary
        GROUP BY store_name, location
        ORDER BY revenue DESC
    ''')
)


In [ ]:
# Category performance
display(
    spark.sql('''
        SELECT
            category,
            ROUND(SUM(total_sales_amount), 2) AS revenue,
            SUM(total_quantity_sold) AS units_sold
        FROM retail_workspace.default.retail_gold_sales_summary
        GROUP BY category
        ORDER BY revenue DESC
    ''')
)


## 10. Consumption Layer

The Unity Catalog Gold table `retail_workspace.default.retail_gold_sales_summary`
is consumed in Power BI through a Databricks Serverless SQL Warehouse.
